# Make sure the rotation matrices correspond to the definition used in EBSD

In [1]:
import numpy as np
from scipy.spatial.transform import Rotation as rot

In [2]:
q_0 = rot.from_euler('Z', -10, degrees=True)
q_0.as_matrix()

array([[ 0.98480775,  0.17364818,  0.        ],
       [-0.17364818,  0.98480775, -0.        ],
       [-0.        ,  0.        ,  1.        ]])

In [3]:
q_0 = rot.from_euler('Z', -10, degrees=True)
np.dot(q_0.as_matrix(), np.array([1,0,0]))

array([ 0.98480775, -0.17364818,  0.        ])

In [4]:
# Validation stuff
# Given in Bunge notation as phi1,Phi,phi2
bunge_angles = np.array([10,20,130])
#  φ1 (phi1): Rotation angle about the Z-axis of the standard reference frame.
#  Φ (Phi): Rotation angle about the X-axis of the intermediate frame obtained after the first rotation.
#  φ2 (phi2): Rotation angle about the Z-axis of the final frame obtained after the first two rotations.

q_0 = rot.from_euler('Z', -bunge_angles[0], degrees=True)
q_1 = rot.from_euler('X', -bunge_angles[1], degrees=True)
q_2 = rot.from_euler('Z', -bunge_angles[2], degrees=True)

# Thus the full rotation from the reference into the crystal system is given by
# [a,b,c] =  q_z(phi2) * q_x(Phi) * q_z(phi1)    * [x,y,z] and reverse
# [x,y,z] = [q_z(phi2) * q_x(Phi) * q_z(phi1)]^T * [a,b,c]
# Lets define R = [q_z(phi2) * q_x(Phi) * q_z(phi1)]^T
q_ref = q_2*q_1*q_0
r_ref = np.array(q_ref.as_matrix()).T
print(r_ref)

[[-0.75802222 -0.64951905  0.05939117]
 [ 0.59729133 -0.72786853 -0.33682409]
 [ 0.26200263 -0.21984631  0.93969262]]


In [5]:
r_direct = rot.from_euler('ZXZ', bunge_angles, degrees=True)
r_matrix = np.array(r_direct.as_matrix())
print(r_matrix)

[[-0.75802222 -0.64951905  0.05939117]
 [ 0.59729133 -0.72786853 -0.33682409]
 [ 0.26200263 -0.21984631  0.93969262]]


In [6]:
# Transformation of e.g. the material-specific diffusion tensor into the reference space is given by
# D_xyz = R * D_abc * R^T
diff_abc = np.array([[2,0,0],[0,1,0],[0,0,5]])
diff_xyz = r_matrix @ diff_abc @ r_matrix.T
print(diff_abc)
print(diff_xyz)

[[2 0 0]
 [0 1 0]
 [0 0 5]]
[[ 1.58870693 -0.53277761  0.02463398]
 [-0.53277761  1.8105588  -1.10955254]
 [ 0.02463398 -1.10955254  4.60073426]]


### Test intuition for fairly simple rotation

In [7]:
bunge_angles = [45,90,0]
r_direct = rot.from_euler('ZXZ', bunge_angles, degrees=True)
r_matrix = np.array(r_direct.as_matrix())
print(np.dot(r_matrix, np.dot(diff_abc, r_matrix.T)))
print()
print(np.dot(r_matrix, np.dot(diff_abc, r_matrix.T))@ [1,0,0])
print(np.dot(r_matrix, np.dot(diff_abc, r_matrix.T))@ [1,-1,0])
print(np.dot(r_matrix, np.dot(diff_abc, r_matrix.T))@ [0,1,0])

[[ 3.50000000e+00 -1.50000000e+00  6.74023927e-16]
 [-1.50000000e+00  3.50000000e+00 -6.74023927e-16]
 [ 6.74023927e-16 -6.74023927e-16  1.00000000e+00]]

[ 3.50000000e+00 -1.50000000e+00  6.74023927e-16]
[ 5.00000000e+00 -5.00000000e+00  1.34804785e-15]
[-1.50000000e+00  3.50000000e+00 -6.74023927e-16]


### Define yaw, pitch roll for structured agglomerates and then convert to Bunge convention

In [8]:
# Validation stuff
# Given in yaw, pitch, roll notation as alpha, beta, gamma
ypr_angles = np.array([-10,83,-45])

q_0 = rot.from_euler('Z', -ypr_angles[0], degrees=True)
q_1 = rot.from_euler('Y', -ypr_angles[1], degrees=True)
q_2 = rot.from_euler('X', -ypr_angles[2], degrees=True)

# Thus the full rotation from the reference into the crystal system is given by
# [a,b,c] =  q_z(gamma) * q_x(beta) * q_z(alpha)    * [x,y,z] and reverse
# [x,y,z] = [q_z(gamma) * q_x(beta) * q_z(alpha)]^T * [a,b,c]
# Lets define R = [q_z(gamma) * q_x(beta) * q_z(alpha)]^T
q_ref = q_2*q_1*q_0
r_ref = np.array(q_ref.as_matrix()).T
print(r_ref)

[[ 0.12001787 -0.56838584  0.81396145]
 [-0.02116239  0.8182368   0.57449168]
 [-0.99254615 -0.08617464  0.08617464]]


In [9]:
r_direct = rot.from_euler('ZYX', ypr_angles, degrees=True)
r_matrix = np.array(r_direct.as_matrix())
print(r_matrix)

[[ 0.12001787 -0.56838584  0.81396145]
 [-0.02116239  0.8182368   0.57449168]
 [-0.99254615 -0.08617464  0.08617464]]


In [10]:
bunge_angles = np.expand_dims(r_direct.as_euler('ZXZ', degrees=True), axis=0)
print(bunge_angles)

[[125.21433484  85.0564254  -94.96207933]]


Normalize angles to
 - phi1 in [0,180]
 - Phi in [0,180]
 - phi2 in [0,180]

In [11]:
change_signs = bunge_angles[:,0]<0
bunge_angles[change_signs, 0] += 180
bunge_angles[change_signs, 1] *= -1

change_signs = bunge_angles[:,1]<0
bunge_angles[change_signs, 1] += 180
bunge_angles[change_signs, 2] *= -1

change_signs = bunge_angles[:,2]<0
bunge_angles[change_signs, 2] += 180
print(bunge_angles)

[[125.21433484  85.0564254   85.03792067]]


In [12]:
print(rot.from_euler('ZXZ', bunge_angles, degrees=True).as_matrix())

[[[-0.12001787  0.56838584  0.81396145]
  [ 0.02116239 -0.8182368   0.57449168]
  [ 0.99254615  0.08617464  0.08617464]]]


Check if rotated z axis points into the same direction

In [16]:
r_vgl = rot.from_euler('ZXZ', bunge_angles.squeeze(), degrees=True).as_matrix()
print(r_vgl.T[2])
print(r_matrix @ np.array([0,0,1]))

[0.81396145 0.57449168 0.08617464]
[0.81396145 0.57449168 0.08617464]


Check if rotated diffusion tensor is still the same

In [262]:
diff_abc = np.array([[2,0,0],[0,1,0],[0,0,5]])
diff_xyz = r_matrix @ diff_abc @ r_matrix.T
diff_xyz2 = r_vgl @ diff_abc @ r_vgl.T
print(diff_xyz-diff_xyz2)

[[ 4.44089210e-16 -4.44089210e-16  1.08246745e-15]
 [-2.22044605e-16 -8.88178420e-16  5.82867088e-16]
 [ 1.11022302e-15  5.55111512e-16 -4.44089210e-16]]
